In [ ]:
import os
from pathlib import Path
# Resolve submission_partition_draft root from common launch locations.
def _resolve_partition_root():
    cwd = Path(os.getcwd()).resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "primary_script").exists() and (p / "intermediate").exists():
            return p
        candidate = p / "technical_review" / "submission_partition_draft"
        if (candidate / "primary_script").exists():
            return candidate
    raise RuntimeError("Could not locate submission_partition_draft root")
REPO_ROOT = _resolve_partition_root()


In [ ]:
import scvi
import scanpy as sc
import pandas as pd
import os
import numpy as np
pd.set_option('display.max_columns', None)

In [ ]:
os.chdir(str(REPO_ROOT / 'intermediate'))

In [ ]:
for i in range(1, 9):
    print('starting on MM' + str(i))
    adata = sc.read_10x_h5(str(REPO_ROOT / 'primary_dependents' / 'cellranger_h5' / ('10872-MM-' + str(i)) / 'filtered_feature_bc_matrix.h5'))
    adata.var_names_make_unique()
    soc = pd.read_csv(str(REPO_ROOT / 'primary_dependents' / 'souporcell' / ('10872-MM-' + str(i) + '_soc') / 'clusters.tsv'), sep='\t')
    singlet_bc = soc[soc['status'] == 'singlet']['barcode']
    adata_subset = adata[adata.obs.index.isin(singlet_bc), :]
    sc.pp.filter_cells(adata_subset, min_genes = 200)
    sc.pp.filter_genes(adata_subset, min_cells = 10)
    sc.pp.highly_variable_genes(adata_subset, n_top_genes = 5000, subset = True, flavor = 'seurat_v3')
    scvi.model.SCVI.setup_anndata(adata_subset)
    vae = scvi.model.SCVI(adata_subset)
    vae.train()
    solo = scvi.external.SOLO.from_scvi_model(vae)
    solo.train()
    df = solo.predict()
    df['prediction'] = solo.predict(soft = False)
    # df.index = df.index.map(lambda x: x[:-2])
    df.to_csv(str(REPO_ROOT / 'intermediate' / ('solo_scores_MM' + str(i) + '.csv')))
    print('MM' + str(i) + ' solo scores for cells that passed the min_genes threshold and were tagged as singlet by souporcell recorded')